# Keccak Adaptativo Basado en Intentos Fallidos

## Factor Dinámico

Se define un factor dinámico de seguridad:

$$
\alpha = \frac{2^f}{\sqrt{d}}
$$
Se tomo la idea de un factor exponencial, y raiz cuadrada de las dimensiones es un factor de suavización como lo es la temperatura en softmax de la idea de los transformer.
donde:

- $f$ = número de intentos fallidos.
- $d$ = número de dimensiones del estado interno.

---


### Variante Hipercubo

Se añade una dimensión adicional asociada a los intentos fallidos:

$$
5 \times 5 \times z
$$

Por tanto:

$$
d = 3
$$

donde $z$ representa la profundidad dinámica del hipercubo.

---

## Evolución del Factor Dinámico

Para el caso clásico ($d=2$):

$$
\alpha = \frac{2^f}{\sqrt{2}}
$$

Ejemplos:

| Intentos fallidos ($f$) | $\alpha$ |
|------------------------|----------|
| 1 | 1.41 |
| 2 | 2.83 |
| 3 | 5.66 |
| 4 | 11.31 |
| 5 | 22.63 |

El crecimiento es exponencial respecto al número de intentos fallidos.

La raíz cuadrada de las dimensiones actúa como un factor de suavización, similar al papel de la temperatura en Softmax, evitando incrementos excesivamente abruptos.

---

## Modificación del Paso Iota

En Keccak estándar:

$$
A_{0,0} = A_{0,0} \oplus RC_i
$$

donde:

- $A_{0,0}$ es el carril de origen.
- $RC_i$ es la constante de ronda.

En la variante adaptativa:

$$
A_{0,0}
=
A_{0,0}
\oplus
\left(
RC_i
\oplus
\lfloor \alpha \rfloor
\right)
$$

De esta forma, los intentos fallidos modifican dinámicamente las constantes de ronda utilizadas por el algoritmo.

---

## Número Adaptativo de Rondas

Se propone que el número de rondas dependa de $\alpha$:

$$
R = \min\left(24,\;8+\lfloor\alpha\rfloor\right)
$$

donde:

- 8 es el número mínimo de rondas.
- 24 es el máximo permitido por Keccak-f.

Cuando aumentan los intentos fallidos, el algoritmo incrementa automáticamente el costo computacional del hashing.

SALT - defensa arcoiris
Registro:
1. Generar salt usando timestamp en milisegundos.
2. Calcular hash adaptativo.
3. Guardar salt, hash e intentos_fallidos.

In [7]:
import math

def keccak_f1600_adaptativo(
    state,
    intentos_fallidos=0,
    dimensiones=3
):
    """
    Variante adaptativa de Keccak-f1600.
    """

    alpha = (
        2 ** intentos_fallidos
    ) / math.sqrt(dimensiones)

    num_rondas = min(
        24,
        8 + int(alpha)
    )

    for round_idx in range(num_rondas):

        # =========================
        # Theta
        # =========================

        C = [0] * 5

        for x in range(5):
            C[x] = (
                state[x][0]
                ^ state[x][1]
                ^ state[x][2]
                ^ state[x][3]
                ^ state[x][4]
            )

        D = [0] * 5

        for x in range(5):
            D[x] = (
                C[(x - 1) % 5]
                ^
                ROTL64(
                    C[(x + 1) % 5],
                    1
                )
            )

        for x in range(5):
            for y in range(5):
                state[x][y] ^= D[x]

        # =========================
        # Rho + Pi
        # =========================

        next_state = [
            [0] * 5
            for _ in range(5)
        ]

        for x in range(5):
            for y in range(5):

                next_state[
                    y
                ][
                    (2*x + 3*y) % 5
                ] = ROTL64(
                    state[x][y],
                    ROTATION_OFFSETS[x][y]
                )

        state = next_state

        # =========================
        # Chi
        # =========================

        next_state = [
            [0] * 5
            for _ in range(5)
        ]

        for x in range(5):
            for y in range(5):

                next_state[x][y] = (
                    state[x][y]
                    ^
                    (
                        (~state[(x+1)%5][y])
                        &
                        state[(x+2)%5][y]
                    )
                )

        state = next_state

        # =========================
        # Iota Adaptativo
        # =========================

        dynamic_rc = (
            RC[round_idx]
            ^
            int(alpha)
        )

        state[0][0] ^= dynamic_rc

    return state

state = keccak_f1600_adaptativo

In [8]:
def sha3_256_adaptativo(
    message: bytes,
    intentos_fallidos=0
):
    rate_bytes = 136

    state = [[0] * 5 for _ in range(5)]

    padded = bytearray(message)
    padded.append(0x06)

    while len(padded) % rate_bytes != 0:
        padded.append(0x00)

    padded[-1] |= 0x80

    for block_idx in range(
        0,
        len(padded),
        rate_bytes
    ):

        block = padded[
            block_idx:
            block_idx + rate_bytes
        ]

        for i in range(rate_bytes // 8):

            x = i % 5
            y = i // 5

            word = int.from_bytes(
                block[i*8:(i+1)*8],
                "little"
            )

            state[x][y] ^= word

        state = keccak_f1600_adaptativo(
            state,
            intentos_fallidos=intentos_fallidos,
            dimensiones=3
        )

    output = bytearray()

    for i in range(4):

        x = i % 5
        y = i // 5

        output.extend(
            state[x][y].to_bytes(
                8,
                "little"
            )
        )

    return bytes(output)

In [9]:
if __name__ == "__main__":
    import hashlib

    casos_prueba = [
        b"",
        b"abc",
        b"El cifrado Keccak cumple con el estandar FIPS 202 sin problemas de compatibilidad.",
        b"a" * 135,  # Frontera de bloque - 1 byte
        b"a" * 136,  # Frontera exacta de un bloque
        b"a" * 137   # Frontera de bloque + 1 byte
    ]

    print("--- VERIFICACIÓN FIPS 202 ---")
    for msg in casos_prueba:
        hash_propio = sha3_256_fips(msg).hex()
        hash_adaptativo_0 = sha3_256_adaptativo(
                            msg,
                            intentos_fallidos=0
                            ).hex()

        hash_adaptativo_3 = sha3_256_adaptativo(
                            msg,
                            intentos_fallidos=3
                            ).hex()

        hash_adaptativo_5 = sha3_256_adaptativo(
                            msg,
                            intentos_fallidos=5
                            ).hex()
        hash_oficial = hashlib.sha3_256(msg).hexdigest()

        print(f"Mensaje len({len(msg)}): {msg[:20]}...")
        print(f"  > Propio:  {hash_propio}")
        print(f"  > hash_adaptativo_0:  {hash_adaptativo_0}")
        print(f"  > hash_adaptativo_3:  {hash_adaptativo_3}")
        print(f"  > hash_adaptativo_5:  {hash_adaptativo_5}")

        print(f"  > Oficial: {hash_oficial}")

        assert hash_propio == hash_oficial, f"¡Error : {msg}!"

    print("\n¡hash  coinciden  con el estándar FIPS.")


--- VERIFICACIÓN FIPS 202 ---
Mensaje len(0): b''...
  > Propio:  a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a
  > hash_adaptativo_0:  6d6f17b37e24db54364a82af3bc7aed23e49438b3e424af8b06056d84f726023
  > hash_adaptativo_3:  0584cbadda193c672d3aadd514acf2440516d9c2d2572b4bef19a0b1303e1e8b
  > hash_adaptativo_5:  915bcfcf8d4918ec0f1f6e6293aa0e6f63861d3bdc3e04421cd05abfc22c4ee3
  > Oficial: a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a
Mensaje len(3): b'abc'...
  > Propio:  3a985da74fe225b2045c172d6bd390bd855f086e3e9d525b46bfe24511431532
  > hash_adaptativo_0:  2bb8589a5fd1131434c11e92a7d2fdc49b2eafc0de32992ffc4d6ad0d8b58301
  > hash_adaptativo_3:  c51691cd003ea056c1996257c49d1c7d4d31a469bdf68fba4470efac5e0d80ae
  > hash_adaptativo_5:  9374b079797f1c9fc8ea092cad18e2b6efa69c80b31b79628fcc6c0f11ae21ea
  > Oficial: 3a985da74fe225b2045c172d6bd390bd855f086e3e9d525b46bfe24511431532
Mensaje len(82): b'El cifrado Keccak cu'...
  > Propio:  75b132c9693574d34d

In [10]:
#con salt
import time

password = "clave123"

salt = str(
    int(time.time() * 1000)
)

print("Salt:", salt)

mensaje = (
    salt + password
).encode()

hash_result = sha3_256_adaptativo(
    mensaje,
    intentos_fallidos=2
)

print(
    hash_result.hex()
)

Salt: 1781239388265
67df17606198f975427e11ce078508d2fd2a3caa97339b8c7204b42806adff1c
